In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os
import pandas as pd

path ='/content/drive/MyDrive/Google_AI_Studio/PFA_'
print(os.listdir(path))

['Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']


In [11]:
import os
import pandas as pd
import glob

path = '/content/drive/MyDrive/Google_AI_Studio/PFA_/'
all_files = glob.glob(path + '*.csv')

# Lire avec encodage latin-1
df = pd.concat(
    [pd.read_csv(f, encoding='latin-1') for f in all_files],
    ignore_index=True
)

print(df.shape)
print(df.head())
print(df.columns.tolist())

/tmp/ipykernel_5193/1853831950.py:10: DtypeWarning: Columns (0,1,3,6,84) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f, encoding='latin-1') for f in all_files],


(3119345, 85)
                                  Flow ID      Source IP   Source Port  \
0   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126          80.0   
1   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126          80.0   
2   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126          80.0   
3   192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126          80.0   
4  192.168.10.14-8.253.185.121-49486-80-6  8.253.185.121          80.0   

   Destination IP   Destination Port   Protocol            Timestamp  \
0    192.168.10.5            49188.0        6.0  03/07/2017 08:55:58   
1    192.168.10.5            49188.0        6.0  03/07/2017 08:55:58   
2    192.168.10.5            49188.0        6.0  03/07/2017 08:55:58   
3    192.168.10.5            49188.0        6.0  03/07/2017 08:55:58   
4   192.168.10.14            49486.0        6.0  03/07/2017 08:56:22   

    Flow Duration   Total Fwd Packets   Total Backward Packets  ...  \
0             4.0                 2.0

In [12]:
# 1. Voir la distribution des labels
print(df[' Label'].value_counts())

 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack  Brute Force         1507
Web Attack  XSS                  652
Infiltration                       36
Web Attack  Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [13]:
# 2. Nettoyer les noms de colonnes (espaces en trop)
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH 

In [14]:
# 3. Supprimer les colonnes non utiles pour le ML
df = df.drop(columns=['Flow ID', 'Source IP', 'Destination IP', 'Timestamp'])

# 4. Vérifier les valeurs manquantes
print(df.isnull().sum().sum(), "valeurs manquantes")

# 5. Supprimer les lignes avec NaN et infinis
import numpy as np
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print(df.shape)

23378120 valeurs manquantes
(2827876, 81)


In [15]:
# 6. Séparer features (X) et label (y)
X = df.drop(columns=['Label'])
y = df['Label']

print("Features:", X.shape)
print("Labels:", y.value_counts())

Features: (2827876, 80)
Labels: Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack  Brute Force         1507
Web Attack  XSS                  652
Infiltration                       36
Web Attack  Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [16]:
# 7. Encoder les labels (BENIGN=0, attaque=1 ou multi-classe)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(le.classes_)

['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack \x96 Brute Force'
 'Web Attack \x96 Sql Injection' 'Web Attack \x96 XSS']
